# 01 Are the data and splits sound?

#### Execution Notes: 

- This notebook downloads NFCorpus from the Hugging Face Hub, embeds it, and computes every number from scratch. NFCorpus repo and qrels revisions match `src.silent_rag.data SOURCES['nfcorpus']`
- It does not read `results/`, `cache/`, or any `silent_rag` module, so it runs on a fresh machine with only the pip packages below. Expect roughly 10-20 minutes on a laptop CPU (embedding ~3.6k documents, the 10,000-permutation nearest-neighbour tests and the AUROC permutation test).

**Questions**
1. Can we tell train, validation and test queries apart?
2. Are there near-duplicate queries or shared relevant documents across splits, and do they inflate scores?
3. Does a query-independent popularity ranking already score well?
4. Do empty documents distort retrieval? Do embeddings carry information in vector length?

**Approach**
Classifier AUROC against a permutation null, with an equivalence check; size-matched nearest-neighbour test with Monte-Carlo intervals and a duplicate-free sensitivity run; duplicate counts at cosine 0.90/0.95/0.98; popularity-prior baseline; norms measured before each model's Normalize layer. Each analysis section states why its test was chosen ("Why this test"). All labelled queries are used, including the Hub's `validation` qrels split.

**Scope.** NFCorpus only. It is the project's development set. SciFact and FiQA are held out for the final run, so this notebook deliberately does not load them.

In [ ]:
# Uncomment on a fresh environment.
# %pip install -q datasets==5.1.0 sentence-transformers==6.1.0 torch==2.14.1 scikit-learn==1.9.1 scipy==1.18.1 pandas==3.0.6 numpy==2.5.4 matplotlib=3.11.2

In [ ]:
import os
# macOS: faiss/torch OpenMP clash; harmless elsewhere
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  
os.environ.setdefault("OMP_NUM_THREADS", "1")

import platform
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

SEED = 0
np.random.seed(SEED)
K = 10
SPLITS = ["train", "validation", "test"]  # the Hub qrels name is "validation", not "dev"



# NFCorpus, pinned to Hub revisions so the numbers are reproducible.
# revisions match src.silent_rag.data SOURCES['nfcorpus']
NFCORPUS = dict(
    repo="BeIR/nfcorpus", revision="b5026a0e96e8a7ac4f95f482a596389289d46269",
    qrels="BeIR/nfcorpus-qrels", qrels_revision="a451b3b26d3ae1358f259c1a3a4dd61fcea35a65",
)
MAX_SEQ = 512
MAIN_MODEL = "multi-qa-MiniLM"
MODELS = {
    "multi-qa-MiniLM": dict(repo="sentence-transformers/multi-qa-MiniLM-L6-cos-v1", revision="b207367332321f8e44f96e224ef15bc607f4dbf0", query_prefix="", doc_prefix=""),
    "all-MiniLM": dict(repo="sentence-transformers/all-MiniLM-L6-v2", revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41", query_prefix="", doc_prefix=""),
    "bge-small": dict(repo="BAAI/bge-small-en-v1.5", revision="5c38ec7c405ec4b44b94cc5a9bb96e735b38267a", query_prefix="Represent this sentence for searching relevant passages: ", doc_prefix=""),
    "e5-small": dict(repo="intfloat/e5-small-v2", revision="ffb93f3bd4047442299a41ebb6fa998a38507c52", query_prefix="query: ", doc_prefix="passage: "),
}

## Helper functions

Everything the analysis needs is defined here: 
- data loading
- encoding
- exact cosine search
- nDCG@10 (graded gain, log2 discount, as in BEIR).

In [ ]:
def doc_text(title, text):
    return f"{(title or '').strip()} {(text or '').strip()}".strip()


def norm_text(t):
    """Case- and whitespace-folded query text, used to find exact duplicate queries."""
    return " ".join(t.lower().split())


def load_nfcorpus():
    """Corpus plus every labelled query from train, validation and test qrels.
    A query keeps the first split it appears in."""
    c = NFCORPUS
    corpus = load_dataset(c["repo"], "corpus", split="corpus", revision=c["revision"])
    queries = load_dataset(c["repo"], "queries", split="queries", revision=c["revision"])
    qtext = {str(q["_id"]): q["text"] for q in queries}
    qrels_by_split = {}
    for split in SPLITS:
        rows = load_dataset(c["qrels"], split=split, revision=c["qrels_revision"])
        q = {}
        for row in rows:
            q.setdefault(str(row["query-id"]), {})[str(row["corpus-id"])] = int(row["score"])
        qrels_by_split[split] = q
    split_of, qrels = {}, {}
    for split in SPLITS:
        for qid, rels in qrels_by_split[split].items():
            if qid not in split_of:
                split_of[qid] = split
                qrels[qid] = rels
    doc_ids = [str(d["_id"]) for d in corpus]
    docs = [doc_text(t, x) for t, x in zip(corpus["title"], corpus["text"])]
    return dict(doc_ids=doc_ids, docs=docs, queries={q: qtext[q] for q in qrels},
                qrels=qrels, split_of=split_of, qrels_by_split=qrels_by_split, n_hub_queries=len(qtext))


def load_encoder(label, max_seq=MAX_SEQ):
    spec = MODELS[label]
    enc = SentenceTransformer(spec["repo"], revision=spec["revision"], device="cpu")
    enc.max_seq_length = max_seq
    return enc


def encode(enc, texts, batch_size=32):
    return np.ascontiguousarray(enc.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)


def search(D, Q, depth=100):
    """Exact top-`depth` by inner product (vectors are unit length, so this is cosine)."""
    S = Q @ D.T
    # Stable full sort: exact score ties (the corpus contains duplicate documents) break by document index,
    # so the top-k does not depend on which search routine is used.
    top = np.argsort(-S, axis=1, kind="stable")[:, :depth]
    return top, np.take_along_axis(S, top, axis=1)


def ndcg_at_k(ranked, rels, k=K):
    dcg = sum(rels.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted(rels.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def perm_p_mc_interval(n_extreme, n_perm, alpha=0.05):
    """Clopper-Pearson interval for the true permutation p-value, given n_extreme of n_perm null draws were at
    least as extreme. The reported p is (1 + n_extreme) / (1 + n_perm); this interval is its Monte-Carlo error."""
    lo = stats.beta.ppf(alpha / 2, n_extreme, n_perm - n_extreme + 1) if n_extreme > 0 else 0.0
    hi = stats.beta.ppf(1 - alpha / 2, n_extreme + 1, n_perm - n_extreme)
    return float(lo), float(hi)


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    out, run = np.empty(len(p)), 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, p[i] * (len(p) - rank)))
        out[i] = run
    return out

## How many queries are in each split

How many labelled queries each split holds (including `validation`), against the total number of queries on the Hub.

In [ ]:
data = load_nfcorpus()
doc_ids, docs, qrels, split_of = data["doc_ids"], data["docs"], data["qrels"], data["split_of"]
qrels_by_split = data["qrels_by_split"]

counts = pd.Series(split_of).value_counts()
overlaps = {f"{a}&{b}": len(set(qrels_by_split[a]) & set(qrels_by_split[b]))
            for i, a in enumerate(SPLITS) for b in SPLITS[i + 1:]}
no_pos = sum(not any(r > 0 for r in rels.values()) for rels in qrels.values())
pd.DataFrame([{"dataset": "nfcorpus", "docs": len(docs), "queries_on_hub": data["n_hub_queries"],
               "labelled": len(qrels), **{s: int(counts.get(s, 0)) for s in SPLITS},
               "no_positive_qrel": no_pos, "split_overlap": overlaps}])

In [ ]:
# Embed the corpus and every labelled query once with the main model, then run clean retrieval.
enc = load_encoder(MAIN_MODEL)
spec = MODELS[MAIN_MODEL]
D = encode(enc, [spec["doc_prefix"] + t for t in docs])

all_qids = list(data["queries"])
keep = [i for i, q in enumerate(all_qids) if any(r > 0 for r in qrels[q].values())]
qids = [all_qids[i] for i in keep]
Q = encode(enc, [spec["query_prefix"] + data["queries"][q] for q in qids])

top, top_scores = search(D, Q, depth=100)
ids = [[doc_ids[j] for j in row] for row in top]
ndcg = np.array([ndcg_at_k(r[:K], qrels[q]) for r, q in zip(ids, qids)])
rel = [{d for d, v in qrels[q].items() if v > 0} for q in qids]
sp = np.array([split_of[q] for q in qids])

print({s: (int((sp == s).sum()), round(float(ndcg[sp == s].mean()), 3)) for s in SPLITS}, "(n queries, mean nDCG@10)")

## Can a classifier tell the splits apart?

**Question.** Can a classifier tell which split a query came from, using only its embedding?

**Method.** PCA to 50 dimensions (unsupervised, fit on both groups), then logistic regression with class weights, 5-fold stratified cross-validated AUROC averaged over 3 repeats. The null distribution comes from permuting the split labels (200 permutations, each a full cross-validated AUROC **averaged over 3 repeats, exactly like the observed value**), and the p-value is `(1 + #{null >= observed}) / (1 + B)`. The smallest AUROC the test could flag at 5% is the 95th percentile of the null. If the observed AUROC is below that, the test cannot distinguish the splits at this sample size, which is weaker than showing they are the same. An **equivalence check** then asks the stronger question: is the AUROC inside `(0.45, 0.55)` with 90% confidence (two one-sided 5% tests)?

**Why this test**
- **Estimand:** how well a query's embedding predicts its split, as cross-validated AUROC (0.5 = indistinguishable).
- **Unit of independence:** the query for the permutation (labels are exchangeable across queries under the null that the splits are random draws from one pool). The bootstrap CI also resamples queries, so it ignores topic dependence and is a lower bound on the uncertainty.
- **Assumptions:** exchangeability of split labels under the null; the null must be computed with the *same* statistic as the observed value.
- **Why not the obvious alternative:** a two-sample test on each embedding dimension (384 tests) would need heavy multiplicity correction and misses multivariate differences; a classifier summarises all of them in one statistic. AUROC is threshold-free and insensitive to the unequal split sizes.
- **Resolution / Monte-Carlo error:** with B = 200 the smallest attainable p is 1/201 ≈ 0.005. The equivalence CI comes from a query bootstrap of the averaged out-of-fold scores (2,000 draws); it does not include the variation from refitting the classifier, so it is approximate.

In [ ]:
def cv_scores(X, y, seed, repeats=3):
    """Out-of-fold decision scores for each CV repeat (one row per repeat)."""
    out = []
    for r in range(repeats):
        cv = StratifiedKFold(5, shuffle=True, random_state=seed + r)
        out.append(cross_val_predict(LogisticRegression(max_iter=2000, class_weight="balanced"), X, y, cv=cv, method="decision_function"))
    return np.array(out)


def cv_auc(X, y, seed, repeats=3):
    return float(np.mean([roc_auc_score(y, s) for s in cv_scores(X, y, seed, repeats)]))


B = 200
REPEATS = 3          # the same number of CV repeats for the observed statistic and for every null draw
EQ_MARGIN = 0.05     # equivalence region: AUROC in (0.45, 0.55)
shift_rows, nulls = [], {}
for i, a in enumerate(SPLITS):
    for b in SPLITS[i + 1:]:
        ma, mb = sp == a, sp == b
        X = PCA(50, random_state=SEED).fit_transform(np.vstack([Q[ma], Q[mb]]))
        y = np.r_[np.zeros(ma.sum()), np.ones(mb.sum())]
        scores_obs = cv_scores(X, y, SEED, REPEATS)
        obs = float(np.mean([roc_auc_score(y, s) for s in scores_obs]))
        rng = np.random.default_rng(SEED)
        null = np.array([cv_auc(X, rng.permutation(y), SEED + 100 + REPEATS * k, repeats=REPEATS) for k in range(B)])
        nulls[(a, b)] = (obs, null)
        # approximate CI: bootstrap queries on the repeat-averaged out-of-fold score (classifier refits not included)
        s_bar = scores_obs.mean(axis=0)
        brng = np.random.default_rng(SEED)
        boot = []
        for _ in range(2000):
            ix = brng.integers(0, len(y), len(y))
            if y[ix].min() != y[ix].max():
                boot.append(roc_auc_score(y[ix], s_bar[ix]))
        lo90, hi90 = np.percentile(boot, [5, 95])
        lo95, hi95 = np.percentile(boot, [2.5, 97.5])
        shift_rows.append({"pair": f"{a} vs {b}", "n_a": int(ma.sum()), "n_b": int(mb.sum()), "auroc": obs,
                           "null_mean": null.mean(), "null_sd": null.std(ddof=1), "null_p95": np.percentile(null, 95),
                           "p_value": (1 + (null >= obs).sum()) / (1 + B), "ci95_lo": lo95, "ci95_hi": hi95,
                           "ci90_lo": lo90, "ci90_hi": hi90,
                           "equivalent_within_0.05": bool(lo90 > 0.5 - EQ_MARGIN and hi90 < 0.5 + EQ_MARGIN)})
shift = pd.DataFrame(shift_rows)

fig, axes = plt.subplots(1, len(nulls), figsize=(3.2 * len(nulls), 2.8), squeeze=False)
for ax, ((a, b), (obs, null)) in zip(axes[0], nulls.items()):
    ax.hist(null, bins=20, color="lightgrey")
    ax.axvline(obs, color="tab:red")
    ax.set_title(f"{a} vs {b}", fontsize=8)
    ax.set_xlabel("AUROC (mean of 3 CV repeats)")
fig.tight_layout()
plt.show()
shift.round(3)

## Are queries in one split unusually close to another split?

For each query in split B, compare its nearest neighbour inside B (all other B queries) with its nearest neighbour in a random subset of split A of the **same size** (`|B| - 1`). If the splits are exchangeable, the difference is centred on what the permutation null gives. Comparing against the whole train split would favour train by construction (it is about 8 times larger than test).

**Statistic (exact, no Monte-Carlo subsets).** The expected nearest-neighbour similarity to a random size-`m` subset of `n` queries is computed analytically: sort a query's similarities to A in descending order `s_(1) >= s_(2) >= ...`; the largest similarity in a random subset is `s_(k)` with probability `C(n-k, m-1) / C(n, m)`, so the expectation is the weighted sum. The weights decay geometrically in `k`, so only the top `L` similarities are needed, where `L` is the smallest rank at which the weights sum to `1 - 1e-12` (exact to 12 decimals, and much faster). The per-query statistic is that expectation minus the within-B nearest neighbour; the mean and the median over B queries are tested.

**Test.** A permutation test (10,000 permutations): pool the A and B queries, shuffle the split labels, recompute **the same statistic**, and compare with the null. Median p-values get a Holm correction across the three pairs.

**Why this test**
- **Estimand:** the mean (and median) over B queries of "expected nearest-neighbour similarity to a same-size A subset minus nearest-neighbour similarity within B". Positive values mean B queries are closer to A than to each other.
- **Unit of independence:** the split label of a query. Per-query differences are *not* independent (the same neighbours serve many queries), so the inference comes from permuting labels, not from treating queries as samples.
- **Assumptions:** exchangeability of labels under the null; observed and permuted statistics computed identically (guaranteed by the exact expectation).
- **Why not the obvious alternative:** a bootstrap or Wilcoxon test over queries would treat the per-query differences as independent and give p-values that are too small. An energy or MMD two-sample test is a valid alternative but answers "any distributional difference", not the closeness question here.
- **Resolution / Monte-Carlo error:** the floor is 1/10,001. Each p-value is reported with a 95% Clopper-Pearson interval for its Monte-Carlo error, and the Holm adjustment is also applied to the upper end of that interval. A result counts as significant only if the Holm-adjusted *upper* end is below 0.05.
- **Sensitivity:** the test is repeated after removing every query whose case-folded text appears more than once (exact duplicates), because duplicates that cross splits are a plausible driver of any excess closeness.

In [ ]:
from scipy.special import gammaln


def expected_max_weights(n, m, tol=1e-12):
    """P(the largest similarity of a random m-subset of n items is the k-th largest of all n), k = 1..L,
    truncated at the smallest L whose weights sum to 1 - tol (the weights decay geometrically in k)."""
    k = np.arange(1, n - m + 2)
    logw = (gammaln(n - k + 1) - gammaln(m) - gammaln(n - k - m + 2)) - (gammaln(n + 1) - gammaln(m + 1) - gammaln(n - m + 1))
    w = np.exp(logw)
    L = min(len(w), int(np.searchsorted(np.cumsum(w), 1 - tol)) + 1)
    return w[:L]


def nn_stat_from_sim(P, ia, ib):
    """Statistic for split A = rows ia, split B = rows ib of the pooled similarity matrix P."""
    n, m = len(ia), len(ib) - 1
    w = expected_max_weights(n, m)
    L = len(w)
    C = P[np.ix_(ib, ia)]
    topL = -np.sort(-np.partition(C, n - L, axis=1)[:, n - L:], axis=1) if L < n else -np.sort(-C, axis=1)
    cross = topL[:, :L] @ w                     # expected NN similarity to a random size-m subset of A
    X = P[np.ix_(ib, ib)].copy()
    np.fill_diagonal(X, -np.inf)
    within = X.max(axis=1)
    d = cross - within
    return float(d.mean()), float(np.median(d)), within, cross


def nn_test(Qa, Qb, nperm, seed=SEED):
    pooled = np.vstack([Qa, Qb])
    P = pooled @ pooled.T
    na = len(Qa)
    obs, med, within, cross = nn_stat_from_sim(P, np.arange(na), np.arange(na, len(pooled)))
    rng = np.random.default_rng(seed)
    null = np.empty((nperm, 2))
    for t in range(nperm):
        idx = rng.permutation(len(pooled))
        null[t] = nn_stat_from_sim(P, idx[:na], idx[na:])[:2]
    row = {"n_A": na, "n_B": len(Qb), "median_within_B": np.median(within), "median_to_A_subset": np.median(cross)}
    for j, (lab, o) in enumerate((("mean", obs), ("median", med))):
        dev = np.abs(null[:, j] - null[:, j].mean())
        n_ext = int((dev >= abs(o - null[:, j].mean())).sum())
        row[f"{lab}_diff"] = o
        row[f"{lab}_null_mean"] = null[:, j].mean()
        row[f"{lab}_z"] = (o - null[:, j].mean()) / null[:, j].std(ddof=1)
        row[f"{lab}_perm_p"] = (1 + n_ext) / (1 + nperm)
        row[f"{lab}_p_mc_lo"], row[f"{lab}_p_mc_hi"] = perm_p_mc_interval(n_ext, nperm)
    return row


NPERM = 10_000
PAIRS = [("train", "test"), ("validation", "test"), ("train", "validation")]
texts_all = np.array([norm_text(data["queries"][q]) for q in qids])
dup_text = pd.Series(texts_all).duplicated(keep=False).to_numpy()     # every member of an exact-duplicate group

nn_tables = {}
for variant, keep_mask in (("all queries", np.ones(len(qids), bool)), ("exact duplicates removed", ~dup_text)):
    rows = []
    for a, b in PAIRS:
        rows.append({"variant": variant, "A": a, "B": b, **nn_test(Q[(sp == a) & keep_mask], Q[(sp == b) & keep_mask], NPERM)})
    t = pd.DataFrame(rows)
    for lab in ("mean", "median"):
        t[f"{lab}_p_holm"] = holm(t[f"{lab}_perm_p"])
        t[f"{lab}_p_holm_mc_hi"] = holm(t[f"{lab}_p_mc_hi"])     # Holm applied to the upper Monte-Carlo bound
        t[f"{lab}_significant_robust"] = t[f"{lab}_p_holm_mc_hi"] < 0.05
    nn_tables[variant] = t
nn_table = nn_tables["all queries"]
print("queries in an exact-duplicate text group:", int(dup_text.sum()), "| by split:",
      {s: int((dup_text & (sp == s)).sum()) for s in SPLITS})
cols = ["variant", "A", "B", "n_A", "n_B", "mean_diff", "mean_z", "mean_perm_p", "mean_p_mc_lo", "mean_p_mc_hi", "mean_p_holm", "mean_p_holm_mc_hi",
        "mean_significant_robust", "median_diff", "median_perm_p", "median_p_mc_lo", "median_p_mc_hi", "median_p_holm", "median_p_holm_mc_hi"]
pd.concat(nn_tables.values(), ignore_index=True)[cols].round(4)

## Near-duplicate queries and shared relevant documents

Three cosine thresholds (0.90, 0.95, 0.98) show how sensitive the counts are. For each, the share of test queries whose nearest **train** query is at least that similar. Separately, the share of test queries that have a relevant document also relevant to some train query (relevant-document overlap), which matters if train queries are used for tuning. Effects on nDCG use a Mann-Whitney test and a bootstrap CI of the mean difference.

**Why this test**
- **Estimand:** the difference in mean test nDCG@10 between test queries that have a train near-duplicate (or share a relevant document with train) and those that do not.
- **Unit of independence:** the test query (an approximation; test queries on one topic are dependent).
- **Assumptions:** the two groups are independent samples. Mann-Whitney tests stochastic dominance, not a difference in means or medians, because nDCG is bounded, zero-inflated and heavily tied; the bootstrap CI is what describes the size of the mean difference.
- **Why not the obvious alternative:** a t-test assumes approximately normal group means, which is doubtful for a group of 6 queries; neither test is run when a group has fewer than 10 queries, because no test has useful power there.
- **Resolution / Monte-Carlo error:** 2,000 bootstrap draws (percentile error about ±0.005 on each CI end). When a comparison group is empty or tiny the effect is reported as untested, not as zero.

In [ ]:
texts = [norm_text(data["queries"][q]) for q in qids]
exact = len(texts) - len(set(texts))
S = Q @ Q.T
np.fill_diagonal(S, -np.inf)
mtest, mtrain = sp == "test", sp == "train"
nn_train = S[np.ix_(mtest, mtrain)].max(axis=1)

train_docs = set().union(*[r for r, s in zip(rel, sp) if s == "train"])
shares_doc = np.array([bool(r & train_docs) for r, s in zip(rel, sp) if s == "test"])

pair_sim = S[np.triu_indices(len(Q), 1)]
dups = pd.DataFrame([{"tau": tau, "pairs_ge_tau": int((pair_sim >= tau).sum()),
                      "test_with_train_nn_ge_tau": float((nn_train >= tau).mean()),
                      "exact_text_duplicates": exact,
                      "test_sharing_relevant_doc_with_train": float(shares_doc.mean())} for tau in (0.90, 0.95, 0.98)])

nd_test = ndcg[mtest]
impact_rows = []
for label, flag in [("train nn >= 0.95", nn_train >= 0.95), ("shares relevant doc with train", shares_doc)]:
    g1, g0 = nd_test[flag], nd_test[~flag]
    if len(g1) >= 10 and len(g0) >= 10:
        rng = np.random.default_rng(SEED)
        diffs = [g1[rng.integers(0, len(g1), len(g1))].mean() - g0[rng.integers(0, len(g0), len(g0))].mean() for _ in range(2000)]
        impact_rows.append({"group": label, "n_with": len(g1), "n_without": len(g0), "mean_with": g1.mean(), "mean_without": g0.mean(),
                            "diff": g1.mean() - g0.mean(), "ci_lo": np.percentile(diffs, 2.5), "ci_hi": np.percentile(diffs, 97.5),
                            "mannwhitney_p": stats.mannwhitneyu(g1, g0).pvalue})
    else:
        impact_rows.append({"group": label, "n_with": len(g1), "n_without": len(g0), "note": "too few queries in one group to compare"})
display(dups.round(3))
pd.DataFrame(impact_rows).round(3)

In [ ]:
# What a near-duplicate looks like: the most similar query pairs
i, j = np.unravel_index(np.argsort(-S, axis=None)[:40:2], S.shape)
for a, b in list(zip(i, j))[:6]:
    print(round(float(S[a, b]), 3), sp[a], sp[b])
    print("   ", data["queries"][qids[a]][:140])
    print("   ", data["queries"][qids[b]][:140])

## How well does ranking by popularity alone do?

If nearly every document is relevant to some train query, a **popularity prior** (rank documents by how many train queries find them relevant, the same list for every query) may already score non-trivially on the test queries. Then part of "retrieval success" reflects the dataset, not the embedding. Reported with a bootstrap CI over test queries, next to the model's nDCG on the same queries.

**Why this test**
- **Estimand:** mean nDCG@10 on the test queries of one fixed, query-independent top-10 list built from train qrels. It is a label-using *floor* from the dataset prior, not a retrieval method.
- **Unit of independence:** the test query (topic dependence makes the CI somewhat too narrow).
- **Assumptions:** none beyond resampling queries; the percentile bootstrap is used because per-query nDCG of a fixed list is mostly zeros and far from normal.
- **Why not the obvious alternative:** no test against zero is run, because the question is "how large is the floor", not "is it non-zero".
- **Resolution / Monte-Carlo error:** 2,000 bootstrap draws.

In [ ]:
counts_pop = pd.Series(0, index=doc_ids, dtype=int)
for rels in qrels_by_split["train"].values():
    for d, v in rels.items():
        if v > 0 and d in counts_pop.index:
            counts_pop[d] += 1
ranking = list(counts_pop.sort_values(ascending=False, kind="stable").index[:K])
test_q = [q for q, s in zip(qids, sp) if s == "test"]
nd_pop = np.array([ndcg_at_k(ranking, qrels[q]) for q in test_q])
nd_model = ndcg[sp == "test"]

seen = set(counts_pop.index[counts_pop > 0])
test_rel_cov = np.mean([np.mean([d in seen for d in r]) for r, s in zip(rel, sp) if s == "test"])
rng = np.random.default_rng(SEED)
boot = np.array([nd_pop[rng.integers(0, len(nd_pop), len(nd_pop))].mean() for _ in range(2000)])
pd.DataFrame([{"corpus_docs_relevant_to_a_train_query": float((counts_pop > 0).mean()),
               "test_relevant_docs_seen_in_train": float(test_rel_cov),
               "popularity_ndcg10": nd_pop.mean(), "pop_ci_lo": np.percentile(boot, 2.5), "pop_ci_hi": np.percentile(boot, 97.5),
               "model_ndcg10_same_queries": nd_model.mean()}]).round(4)

## Do empty or short documents dominate the results?

Counts the empty (title and text both blank) and very short documents, whether the highest k-occurrence documents are among them, and re-runs retrieval without the empty ones. k-occurrence is how many labelled queries have the document in their top 10. Document length is whitespace words.

**Why this test**
- **Estimand:** the rank correlation between a document's length and its k-occurrence, and the change in mean nDCG@10 when empty documents are removed.
- **Unit of independence:** the document (optimistic: the corpus has 39 identical-vector pairs and topical near-duplicates).
- **Assumptions:** Spearman is used because k-occurrence is a skewed count with many zeros and its relation to length need not be linear; the bootstrap over documents gives the CI.
- **Why not the obvious alternative:** Pearson correlation would be driven by the few heavy hubs.
- **Resolution / Monte-Carlo error:** 500 bootstrap draws (CI ends accurate to about ±0.01).

In [ ]:
words = np.array([len(t.split()) for t in docs])
empty = words == 0
kocc = np.bincount(top[:, :K].ravel(), minlength=len(docs))
rho = stats.spearmanr(words, kocc)[0]
rng = np.random.default_rng(SEED)
boot = [stats.spearmanr(words[s], kocc[s])[0] for s in (rng.integers(0, len(words), len(words)) for _ in range(500))]

if empty.any():
    keepm = ~empty
    kept_ids = [d for d, k in zip(doc_ids, keepm) if k]
    top2, _ = search(D[keepm], Q, depth=K)
    nd2 = np.array([ndcg_at_k([kept_ids[j] for j in row], qrels[q]) for row, q in zip(top2, qids)])
    delta, max_empty_kocc = float(nd2.mean() - ndcg.mean()), int(kocc[empty].max())
else:
    delta, max_empty_kocc = 0.0, 0

display(pd.DataFrame([{"docs": len(words), "empty_docs": int(empty.sum()), "under_10_words": int((words < 10).sum()),
                       "max_kocc_among_empty": max_empty_kocc, "max_kocc_overall": int(kocc.max()),
                       "spearman_kocc_vs_words": rho, "ci_lo": np.percentile(boot, 2.5), "ci_hi": np.percentile(boot, 97.5),
                       "ndcg_change_without_empty": delta}]).round(4))
order = np.argsort(-kocc)[:10]
pd.DataFrame({"doc_id": [doc_ids[i] for i in order], "k_occ": kocc[order], "words": words[order],
              "snippet": [docs[i][:70] for i in order]})

## Do the models store information in vector length?

All four models end in a Normalize layer, so `normalize_embeddings=False` still returns unit vectors (the module chain is in the table). To test whether the model encodes anything in the vector length, the Normalize layer is dropped and the pooled embedding is used directly. 1,000 random NFCorpus documents per model, Spearman correlation of the norm with the document token count, with a 95% bootstrap CI. `cos_prenorm_vs_full` confirms that normalizing the pooled vector reproduces the full model's output. This section downloads the three other models.

**Why this test**
- **Estimand:** per model, the rank correlation between the pre-normalization vector length and the document's token count (descriptive: it says what the Normalize layer discards).
- **Unit of independence:** the document (1,000 sampled without replacement).
- **Assumptions:** monotone, not necessarily linear, relation, hence Spearman; bootstrap CI over documents.
- **Why not the obvious alternative:** no hypothesis test is run; the size and sign of the correlation per model are the content, and no decision depends on rejecting zero.
- **Resolution / Monte-Carlo error:** 1,000 bootstrap draws.

In [ ]:
rng = np.random.default_rng(SEED)
sample = rng.choice(len(docs), 1000, replace=False)
norm_rows = []
for label, spec_ in MODELS.items():
    enc_ = load_encoder(label)
    texts_ = [spec_["doc_prefix"] + docs[i] for i in sample]
    modules = [m for m in enc_ if type(m).__name__ != "Normalize"]
    raw_model = SentenceTransformer(modules=modules, device="cpu")
    full = enc_.encode(texts_[:50], normalize_embeddings=False, convert_to_numpy=True, show_progress_bar=False)
    raw = raw_model.encode(texts_, batch_size=32, normalize_embeddings=False, convert_to_numpy=True, show_progress_bar=False)
    norms = np.linalg.norm(raw, axis=1)
    n_tok = np.array([len(enc_.tokenizer.tokenize(t)) for t in texts_])
    rho_ = stats.spearmanr(norms, n_tok)[0]
    boot_ = [stats.spearmanr(norms[s], n_tok[s])[0] for s in (rng.integers(0, len(norms), len(norms)) for _ in range(1000))]
    cos_check = float(np.mean(np.sum(raw[:50] / np.linalg.norm(raw[:50], axis=1, keepdims=True) * full, axis=1)))
    norm_rows.append({"model": label, "modules": " > ".join(type(m).__name__ for m in enc_),
                      "full_model_norm_sd": float(np.linalg.norm(full, axis=1).std()), "pre_norm_mean": norms.mean(),
                      "pre_norm_sd": norms.std(), "cos_prenorm_vs_full": cos_check, "spearman_norm_vs_tokens": rho_,
                      "ci_lo": np.percentile(boot_, 2.5), "ci_hi": np.percentile(boot_, 97.5)})
pd.DataFrame(norm_rows).round(4)

In [ ]:
import datasets, sentence_transformers, sklearn, scipy, torch
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
       "datasets": datasets.__version__, "seed": SEED, "device": "cpu", "max_seq_length": MAX_SEQ})

## Reproducibility notes

**Two sources of tiny differences:** (1) the corpus contains 39 pairs of identical document vectors, so 29 queries have an exact score tie at rank 10/11, and which document is kept depends on the search routine (mean nDCG@10 ranges 0.29581 to 0.29590 across routines). This notebook breaks ties by document index (stable sort), which makes the top-k deterministic. The project's runner (`pipeline.Retriever`) uses FAISS `IndexFlatIP`, which may break these ties differently. (2) Differences in the third decimal are expected across package versions.

## Findings
Numbers are from a full from-scratch run (seed 0, NFCorpus, multi-qa-MiniLM).

- **No classifier signal between splits, and equivalence only for one pair.** 3,237 labelled queries (train 2,590, validation 324, test 323; no query is in two splits). AUROC 0.465 to 0.487, below the permutation-null 95th percentile in every pair (0.534 to 0.547; p 0.69 to 0.90). Equivalence to chance within ±0.05 AUROC is shown only for train vs validation (90% CI 0.462 to 0.517); for the two pairs involving test it is not shown. Mean nDCG@10 is 0.299 / 0.272 / 0.297.
- **One pair differs on nearest-neighbour closeness:** validation vs test (mean difference +0.029, p = 0.011 with Monte-Carlo interval 0.009 to 0.013, Holm 0.032; robust to Monte-Carlo error). The median version is not significant (p = 0.057). Train vs test and train vs validation do not differ (mean p >= 0.57). **Exact duplicate queries do not explain it:** without the 48 queries in duplicate-text groups the difference is +0.031 (Holm 0.015).
- **Near-duplicates are rare in test but exist.** 4.0% of test queries have a train query at cosine >= 0.90 and 1.9% at >= 0.95; 24 exact (case-folded) duplicate query texts overall (48 queries involved). Too few queries (6) sit above 0.95 to estimate an effect on nDCG.
- **Every test query shares a relevant document with a train query** (100%), and 98.8% of the corpus is relevant to some train query.
- **A single popularity ranking** (the same top 10 for every query, from train qrels) scores nDCG@10 0.073 (95% CI 0.050 to 0.099) against 0.297 for the model on the same test queries.
- **No empty documents** (0 empty, 0 under 10 words). k-occurrence is weakly negatively correlated with length (Spearman -0.16, CI -0.19 to -0.13).
- **Norms.** All four models end in a Normalize layer. Before it, vector length correlates with token count: multi-qa-MiniLM -0.44, all-MiniLM -0.42, bge-small +0.29, e5-small -0.08.

### What this supports
On NFCorpus, a query-embedding classifier finds no difference between the official splits at these sample sizes, and train vs validation is equivalent to chance within ±0.05 AUROC. Validation and test queries sit closer to each other than exchangeable splits would, and exact duplicates do not account for it. A dataset prior alone gets roughly a quarter of the model's nDCG.

### What this does not support
- **That all labelled queries can be pooled as independent evaluation units.** The splits are not distinguishable by a classifier, but queries are not independent: every test query shares a relevant document with train, and 48 queries are exact text duplicates. Pooling increases the sample, but every query-level standard error computed from the pool is too small; group-level (cluster) inference is needed.
- That the splits are *equivalent*: equivalence is shown only for train vs validation, and validation vs test shows excess closeness.
- That duplicates and shared relevant documents do **not** inflate scores: the comparison groups are too small or empty, so the effect is untested.
- That tuning on train and scoring on test is leak-free: every test query shares relevant documents with train.
- Why validation and test are closer to each other: it is not exact duplicates; other drivers are untested.
- Anything about SciFact or FiQA, or about other query sets.